In [ ]:
"""
Optional: Settings to limit this notebook to only a specific GPU
"""
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ['HF_HOME'] = '/media/my_drives/DATA4/models' # Here you can set a path to which Phi-4 from huggingface should be downloaded to. 

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    
if IN_COLAB:
    os.chdir("/content")
    print("Running in Colab. Changed directory to:", os.getcwd())

In [ ]:
import base64
import gc
import glob
import json
import random
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import transformers
import datasets

from PIL import Image
from datasets import Dataset, Features, Value
from evaluate import load
from openai import OpenAI
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoFeatureExtractor,
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

import ast
import pickle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Model training
Run all the following cells to train & predict using ConvNeXt, GPT-5, and the ensemble head

In [ ]:
"""
Settings

You can modify these settings for your training process

"""

MODEL_HUGGINGFACE = 'facebook/convnext-base-384-22k-1k'
MODEL_NAME = 'ConvNeXt'

# Hyperparameter settings
EPOCHS = 32
BATCH_SIZES = [16, 32, 64]
LEARNING_RATES = [1e-2, 1e-3, 1e-5]

OPEN_AI_API_KEY = 'YOUR_OPEN_AI_API_KEY'

# Paths for saving the models and files
MODEL_CACHE_DIR = './HF_training'
OUT_DIR = './Ensemble'
FINAL_MODEL_DIR = os.path.join(OUT_DIR, "Trained_TVM")
FINAL_ENSEMBLE_MODEL_DIR = os.path.join(OUT_DIR, "Trained_Ensemble")
PERFORMANCE_SUMMARY_PATH = f'{OUT_DIR}/training_summary_ensemble_tvm.csv'

# Your dataset with labeled examples
DATASET_PATH = 'Example_Dataset/dataset.csv'

In [ ]:
def create_classification_dataset(DF_DATASET, MODELNAME):
    df = DF_DATASET.copy()
    unique_labels = sorted(df["label"].astype(str).unique().tolist())
    label2id = {label: i for i, label in enumerate(unique_labels)}
    df["labels"] = df["label"].astype(str).map(label2id)
    img_path_list = df.image_path.to_list()
    label_id_list = df.labels.to_list()
    assert len(img_path_list) == len(label_id_list)

    features = datasets.Features({
        "image_path": datasets.Value("string"),
        "img": datasets.Image(),
        "labels": datasets.ClassLabel(names=unique_labels),
    })

    ds = datasets.Dataset.from_dict(
        {
            "img": img_path_list,
            "image_path": img_path_list,
            "labels": label_id_list,
        },
        features=features,
    )
    processor = AutoImageProcessor.from_pretrained(MODELNAME)

    def hf_transform(example_batch):
        inputs = processor(
            [x.convert("RGB") for x in example_batch["img"]],
            return_tensors="pt",
        )
        inputs["labels"] = example_batch["labels"]
        inputs["image_path"] = example_batch["image_path"]
        return inputs

    prepared_ds = ds.with_transform(hf_transform)
    return prepared_ds, label2id, processor

def train_hf_classification_model(outdir, epochs, batch_size, learning_rate, train_dataset, test_dataset, MODEL_NAME):
        def collate_fn(batch):
            return {
            'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
            'labels': torch.tensor([x['labels'] for x in batch])
            }
        def custom_metrics(eval_pred):
            metric1 = load("precision")
            metric2 = load("recall")
            metric3 = load("f1")
            metric4 = load("accuracy")
            
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)
        
            precision = metric1.compute(predictions=predictions, references=labels, average="weighted")["precision"]
            recall = metric2.compute(predictions=predictions, references=labels, average="weighted")["recall"]
            f1 = metric3.compute(predictions=predictions, references=labels, average="weighted")["f1"]
            accuracy = metric4.compute(predictions=predictions, references=labels)["accuracy"]
        
            return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}

        labels = train_dataset.features['labels'].names
        processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
        model = AutoModelForImageClassification.from_pretrained(MODEL_NAME, num_labels=len(labels), ignore_mismatched_sizes=True, id2label={str(i): c for i, c in enumerate(labels)}, label2id={c: str(i) for i, c in enumerate(labels)})
            
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)

        early_stopping_patience_epochs = 3
        early_stopping = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_epochs)
        
        training_args = TrainingArguments(
        output_dir = outdir,
        disable_tqdm=True,
        per_device_train_batch_size=batch_size,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=0.01,   
        save_total_limit=2,
        remove_unused_columns=False,
        push_to_hub=False,
        load_best_model_at_end=True,
        no_cuda=False,
        fp16=True
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            data_collator=collate_fn,
            compute_metrics=custom_metrics,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            tokenizer=processor,
            callbacks=[early_stopping]
        )

        train_results = trainer.train()
        log_history = trainer.state.log_history
        best_epoch = None
        eval_entries = [e for e in log_history if "eval_loss" in e or "eval_accuracy" in e]
        if eval_entries:
            if any("eval_loss" in e for e in eval_entries):
                best_entry = min(
                    (e for e in eval_entries if "eval_loss" in e),
                    key=lambda x: x["eval_loss"],
                )
            else:
                best_entry = max(
                    eval_entries,
                    key=lambda x: x.get("eval_accuracy", float("-inf")),
                )
            best_epoch = best_entry.get("epoch")
        if best_epoch is None:
            best_epoch = trainer.state.epoch
        optimal_epochs_trained = int(round(best_epoch)) if best_epoch is not None else None
        metrics = trainer.evaluate(test_dataset)
        return trainer, model, metrics, optimal_epochs_trained

In [ ]:
""" 

Loading your dataset

"""

df = pd.read_csv(DATASET_PATH)

required_cols = {'image_path', 'label'}
assert required_cols.issubset(df.columns), \
    'Please make sure that your dataset contains both columns: image_path and label.'

df.head()

In [ ]:
""" Make test dataset """

dataset, label2id, processor = create_classification_dataset(df, MODELNAME=MODEL_HUGGINGFACE)
split = dataset.train_test_split(test_size=0.2, seed=1)
train_val_dataset = split["train"]
test_dataset = split["test"]

In [ ]:
"""

Fine-tune the model with your dataset

"""
os.makedirs(OUT_DIR, exist_ok=True)

df_train_summary = pd.DataFrame()
print('Training models to find optimal hyperparameters.')
for LEARNING_RATE in LEARNING_RATES:
    for BATCH_SIZE in BATCH_SIZES:
        ds = train_val_dataset.shuffle(seed=1)

        fold_counter = 0
        kf = KFold(n_splits=10, shuffle=True, random_state=1)
        for train_index, val_index in kf.split(ds):
            fold_counter += 1
            train_dataset = ds.select(train_index)
            val_dataset = ds.select(val_index)
    
            trainer, model, metrics, optimal_epochs_trained = train_hf_classification_model(outdir=MODEL_CACHE_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, train_dataset=train_dataset, test_dataset=val_dataset, MODEL_NAME=MODEL_HUGGINGFACE)

            i = df_train_summary.shape[0]
            df_train_summary.at[i, 'Batch_Size'] = BATCH_SIZE
            df_train_summary.at[i, 'Learning_Rate'] = LEARNING_RATE
            df_train_summary.at[i, 'Fold'] = fold_counter
            df_train_summary.at[i, 'Epochs'] = optimal_epochs_trained
            df_train_summary.at[i, 'Accuracy'] = np.round(metrics['eval_accuracy'], 4)
            df_train_summary.at[i, 'Precision'] = np.round(metrics['eval_precision'], 4)
            df_train_summary.at[i, 'Recall'] = np.round(metrics['eval_recall'], 4)
            df_train_summary.at[i, 'F1'] = np.round(metrics['eval_f1'], 4)
            df_train_summary.to_csv(PERFORMANCE_SUMMARY_PATH, index=False)

print('Training the best model for inference.')
df_avg = (
    df_train_summary
    .groupby(["Batch_Size", "Learning_Rate"], as_index=False)
    .agg(
        AVG_ACCURACY=("Accuracy", "mean"),
        STD_ACCURACY=("Accuracy", "std"),
        N_FOLDS=("Accuracy", "count"),
        AVG_OPTIMAL_EPOCHS=("Epochs", "mean"),
    )
)
df_avg = df_avg.sort_values("AVG_ACCURACY", ascending=False)

best_row = df_avg.iloc[0]
best_batch_size = int(best_row["Batch_Size"])
best_learning_rate = float(best_row["Learning_Rate"])

split = train_val_dataset.train_test_split(test_size=0.2, seed=1)
train_dataset = split["train"]
val_dataset = split["test"]

trainer, model, metrics, optimal_epochs_trained = train_hf_classification_model(outdir=MODEL_CACHE_DIR, epochs=EPOCHS, batch_size=best_batch_size, learning_rate=best_learning_rate, train_dataset=train_dataset, test_dataset=val_dataset, MODEL_NAME=MODEL_HUGGINGFACE)
model.config.label2id = label2id
model.config.id2label = {v: k for k, v in label2id.items()}

trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)


In [ ]:
"""

Use the trained ConvNeXt TVM for prediction on train and val datasets

"""
from PIL import Image
# Split the dataset
split = train_val_dataset.train_test_split(test_size=0.2, seed=1)
train_dataset = split["train"]
val_dataset = split["test"]

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForImageClassification.from_pretrained(FINAL_MODEL_DIR)
processor = AutoImageProcessor.from_pretrained(FINAL_MODEL_DIR)

model.to(device)
model.eval()

def predict_batch(examples):
    """Function to predict class probabilities for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        image = Image.open(image_path).convert('RGB')
        
        # Preprocess
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
        
        # logits → probabilities
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        
        # Convert to dictionary with class labels
        class_probs = {model.config.id2label[i]: np.round(float(probs[0, i]), 6) for i in range(probs.shape[1])}
        predictions.append(json.dumps(class_probs))
    
    return {'prediction_convnext_allclassprobs': predictions}

# Add predictions to train dataset
print("Predicting on train dataset...")
train_dataset = train_dataset.map(predict_batch, batched=True, batch_size=1)

# Add predictions to val dataset
print("Predicting on val dataset...")
val_dataset = val_dataset.map(predict_batch, batched=True, batch_size=1)

In [ ]:
"""

Get predictions using GPT-5

"""
os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY
client = OpenAI()

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_json_class_schema(classes:list):
    def get_list(classes:list):
        output = {}
        for pred_class in classes:
            output[pred_class] = {
            "type": "number",
            "minimum": 0,
            "maximum": 1
          }
        return output
    class_schema = {
    "format": {
      "type": "json_schema",
      "name": "probabilities",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": get_list(classes),
        "required": classes,
        "additionalProperties": False
      }}}
    return class_schema

def predict_gpt(img_path, prompt, classes):
    base64_image = encode_image(img_path)
    class_schema = get_json_class_schema(classes)

    response = client.responses.create(
        model="gpt-5",
        input=[{
            "role": "system",
            "content": "You are an image classification model. Always return valid JSON with class probabilities that sum to 1."
        },
               {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
        text=class_schema
    )
    
    return response.output_text

In [ ]:
"""

Use GPT-5 for prediction on train and val datasets

"""

# Get unique classes from the dataset
classes = list(label2id.keys())
prompt = f'Classify the best fitting class to describe the image by choosing one of the following classes. Return probabilities for all classes in JSON.'

def predict_gpt_batch(examples):
    """Function to predict class probabilities using GPT for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        try:
            response = predict_gpt(image_path, prompt, classes)
            predictions.append(response)
        except Exception as e:
            print(f"Error predicting {image_path}: {e}")
            # Return empty dict in case of error
            predictions.append(json.dumps({c: 0.0 for c in classes}))
    
    return {'prediction_gpt5_allclassprobs': predictions}

# Add GPT predictions to train dataset
print("Predicting on train dataset with GPT-5...")
train_dataset = train_dataset.map(predict_gpt_batch, batched=True, batch_size=1)

# Add GPT predictions to val dataset
print("Predicting on val dataset with GPT-5...")
val_dataset = val_dataset.map(predict_gpt_batch, batched=True, batch_size=1)

In [ ]:
"""

Train the ensemble head

"""

# Convert the prob dicts into a matrix
def dicts_to_matrix(prob_dicts, classes):
    """
    Converts a list of probability dictionaries into a 2D numpy array.
    Each row corresponds to one sample, columns correspond to classes.
    """
    matrix = []
    for d in prob_dicts:
        row = [d.get(c, 0.0) for c in classes]  # fallback to 0 if class not in dict
        matrix.append(row)
    return np.array(matrix)

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, dropout):
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.dropout = nn.Dropout(dropout)
        if n_classes > 1:
            self.fc2 = nn.Linear(256, n_classes)
            self.final_activation = nn.LogSoftmax(dim=1)  # for CrossEntropyLoss, no activation needed, but can still use LogSoftmax
        else:
            self.fc2 = nn.Linear(256, 1)
            self.final_activation = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        if self.final_activation is not None:
            x = self.final_activation(x)
        return x

# Early stopping helper
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.best_model_state = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# Training function
def train_model(model, optimizer, criterion, train_loader, val_loader, max_epochs=100, patience=10, device='cpu'):
    early_stopping = EarlyStopping(patience)
    history = []
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            if output.shape[1] > 1:  # multi-class
                loss = criterion(output, y_batch)
            else:  # binary
                loss = criterion(output.squeeze(), y_batch.float())

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)

                if output.shape[1] > 1:
                    loss = criterion(output, y_batch)
                    preds = torch.argmax(output, dim=1)
                else:
                    loss = criterion(output.squeeze(), y_batch.float())
                    preds = (output.squeeze() > 0.5).long()

                val_loss += loss.item()
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total
        history.append((val_loss, val_acc))

        early_stopping(val_loss, model)

        if early_stopping.early_stop:
            model.load_state_dict(early_stopping.best_model_state)
            break

    return model, history

In [ ]:
"""
Settings

You can modify these settings for your training process

"""

MODEL_NAME = 'Ensemble'
PERFORMANCE_SUMMARY_PATH = f'{OUT_DIR}/training_summary_ensemble.csv'

In [ ]:
df_ensemble_summary = pd.DataFrame()

for BATCH_SIZE in BATCH_SIZES:
    for LEARNING_RATE in LEARNING_RATES:
        # Get class labels from the dataset
        classes = list(label2id.keys())
        
        # Access data via to_dict() to avoid transform issues
        train_dict = train_dataset.to_dict()
        val_dict = val_dataset.to_dict()
        
        # Extract predictions from train_dataset
        gpt_dicts_train = [json.loads(pred) for pred in train_dict['prediction_gpt5_allclassprobs']]
        convnext_dicts_train = [json.loads(pred) for pred in train_dict['prediction_convnext_allclassprobs']]
        
        # Convert dicts to matrices
        X_gpt_train = dicts_to_matrix(gpt_dicts_train, classes)
        X_convnext_train = dicts_to_matrix(convnext_dicts_train, classes)
        
        # Concatenate GPT + ConvNeXt probabilities
        X_train = np.hstack([X_gpt_train, X_convnext_train])
        
        # Labels
        y_train = np.array(train_dict['labels'])
        
        # Extract predictions from val_dataset
        gpt_dicts_val = [json.loads(pred) for pred in val_dict['prediction_gpt5_allclassprobs']]
        convnext_dicts_val = [json.loads(pred) for pred in val_dict['prediction_convnext_allclassprobs']]
        
        # Convert dicts to matrices
        X_gpt_val = dicts_to_matrix(gpt_dicts_val, classes)
        X_convnext_val = dicts_to_matrix(convnext_dicts_val, classes)
        
        # Concatenate GPT + ConvNeXt probabilities
        X_val = np.hstack([X_gpt_val, X_convnext_val])
        
        # Labels
        y_val = np.array(val_dict['labels'])
        
        # Convert to tensors
        input_dim = X_train.shape[1]
        n_classes = len(np.unique(y_train))
        
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.long)
        
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.long)
        
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        val_ds = TensorDataset(X_val_tensor, y_val_tensor)
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        model = SimpleClassifier(input_dim, n_classes, dropout=0.25).to(device)
        
        if n_classes > 1:
            criterion = nn.CrossEntropyLoss()
        else:
            criterion = nn.BCELoss()
        
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
        start_time = time.time()
        model, history = train_model(model, optimizer, criterion, train_loader, val_loader, max_epochs=EPOCHS, patience=10, device=device)
        total_time = np.round(time.time() - start_time, 2)
        
        best_idx = min(range(len(history)), key=lambda i: history[i][0])
        best_epoch = best_idx + 1
        best_val_loss = history[best_idx][0]
        best_val_acc = history[best_idx][1]
        
        index_row = df_ensemble_summary.shape[0]
        df_ensemble_summary.at[index_row, 'Model'] = MODEL_NAME
        df_ensemble_summary.at[index_row, 'Batch_Size'] = BATCH_SIZE
        df_ensemble_summary.at[index_row, 'Learning_Rate'] = LEARNING_RATE
        df_ensemble_summary.at[index_row, 'Epochs'] = best_epoch
        df_ensemble_summary.at[index_row, 'Time_Seconds'] = total_time
        df_ensemble_summary.at[index_row, 'Accuracy'] = best_val_acc
        
        df_ensemble_summary.to_csv(PERFORMANCE_SUMMARY_PATH, index=False)
        
        del X_train, y_train, X_val, y_val
        del model
        gc.collect()

print('Training the best model for inference.')
df_avg = (
    df_ensemble_summary
    .groupby(["Batch_Size", "Learning_Rate"], as_index=False)
    .agg(
        AVG_ACCURACY=("Accuracy", "mean"),
        STD_ACCURACY=("Accuracy", "std"),
        N_FOLDS=("Accuracy", "count")
    )
)
df_avg = df_avg.sort_values("AVG_ACCURACY", ascending=False)

best_row = df_avg.iloc[0]
best_batch_size = int(best_row["Batch_Size"])
best_learning_rate = float(best_row["Learning_Rate"])

# Train final ensemble on full train_dataset (no validation)
classes = list(label2id.keys())

train_dict = train_dataset.to_dict()

gpt_dicts_train = [json.loads(pred) for pred in train_dict['prediction_gpt5_allclassprobs']]
convnext_dicts_train = [json.loads(pred) for pred in train_dict['prediction_convnext_allclassprobs']]

X_gpt_train = dicts_to_matrix(gpt_dicts_train, classes)
X_convnext_train = dicts_to_matrix(convnext_dicts_train, classes)
X_train_full = np.hstack([X_gpt_train, X_convnext_train])
y_train_full = np.array(train_dict['labels'])

input_dim = X_train_full.shape[1]
n_classes = len(np.unique(y_train_full))

X_train_tensor = torch.tensor(X_train_full, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_full, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_ds, batch_size=best_batch_size, shuffle=True)

final_model = SimpleClassifier(input_dim, n_classes, dropout=0.25).to(device)

if n_classes > 1:
    criterion = nn.CrossEntropyLoss()
else:
    criterion = nn.BCELoss()

optimizer = optim.Adam(final_model.parameters(), lr=best_learning_rate)

# Manual training loop without validation
final_model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = final_model(X_batch)
        if output.shape[1] > 1:
            loss = criterion(output, y_batch)
        else:
            loss = criterion(output.squeeze(), y_batch.float())
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f"Final ensemble epoch {epoch + 1}/{EPOCHS} - loss: {avg_loss:.4f}")

os.makedirs(FINAL_ENSEMBLE_MODEL_DIR, exist_ok=True)

label2id_path = os.path.join(OUT_DIR, "label2id.json")
with open(label2id_path, "w") as f:
    json.dump(label2id, f)

# Save the final ensemble model for inference
ensemble_path = os.path.join(FINAL_ENSEMBLE_MODEL_DIR, "ensemble_model.pt")
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "input_dim": input_dim,
        "n_classes": n_classes,
        "dropout": 0.25,
        "label2id": label2id,
        "best_batch_size": best_batch_size,
        "best_learning_rate": best_learning_rate,
    },
    ensemble_path,
)
print(f"Saved ensemble model to {ensemble_path}")

In [ ]:
"""
Load the saved ensemble model and run on test dataset
"""

# Load ensemble checkpoint
ensemble_path = os.path.join(FINAL_ENSEMBLE_MODEL_DIR, "ensemble_model.pt")
checkpoint = torch.load(ensemble_path, map_location=device)

loaded_label2id = checkpoint["label2id"]
loaded_id2label = {v: k for k, v in loaded_label2id.items()}
input_dim = checkpoint["input_dim"]
n_classes = checkpoint["n_classes"]
dropout = checkpoint.get("dropout", 0.25)

ensemble_model = SimpleClassifier(input_dim, n_classes, dropout=dropout).to(device)
ensemble_model.load_state_dict(checkpoint["model_state_dict"])
ensemble_model.eval()

print(f"Loaded ensemble model from {ensemble_path}")

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForImageClassification.from_pretrained(FINAL_MODEL_DIR)
processor = AutoImageProcessor.from_pretrained(FINAL_MODEL_DIR)
model.to(device)
model.eval()

def predict_tvm_batch(examples):
    """Function to predict class probabilities for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        image = Image.open(image_path).convert('RGB')
        
        # Preprocess
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
        
        # logits → probabilities
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        
        # Convert to dictionary with class labels
        class_probs = {model.config.id2label[i]: np.round(float(probs[0, i]), 6) for i in range(probs.shape[1])}
        predictions.append(json.dumps(class_probs))
    
    return {'prediction_convnext_allclassprobs': predictions}

# Get unique classes from the dataset
classes = list(loaded_label2id.keys())
prompt = 'Classify the best fitting class to describe the image by choosing one of the following classes. Return probabilities for all classes in JSON.'

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_json_class_schema(classes:list):
    def get_list(classes:list):
        output = {}
        for pred_class in classes:
            output[pred_class] = {
            "type": "number",
            "minimum": 0,
            "maximum": 1
          }
        return output
    class_schema = {
    "format": {
      "type": "json_schema",
      "name": "probabilities",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": get_list(classes),
        "required": classes,
        "additionalProperties": False
      }}}
    return class_schema

def predict_gpt(img_path, prompt, classes):
    base64_image = encode_image(img_path)
    class_schema = get_json_class_schema(classes)

    response = client.responses.create(
        model="gpt-5",
        input=[{
            "role": "system",
            "content": "You are an image classification model. Always return valid JSON with class probabilities that sum to 1."
        },
               {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
        text=class_schema
    )
    
    return response.output_text

def predict_gpt_batch(examples):
    """Function to predict class probabilities using GPT for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        try:
            response = predict_gpt(image_path, prompt, classes)
            predictions.append(response)
        except Exception as e:
            print(f"Error predicting {image_path}: {e}")
            # Return empty dict in case of error
            predictions.append(json.dumps({c: 0.0 for c in classes}))
    
    return {'prediction_gpt5_allclassprobs': predictions}


# Helper: run ensemble on a HF dataset with prediction columns
@torch.no_grad()
def run_ensemble_inference(hf_dataset, gpt_col="prediction_gpt5_allclassprobs", convnext_col="prediction_convnext_allclassprobs"):
    classes = list(loaded_label2id.keys())

    gpt_dicts = [json.loads(pred) for pred in hf_dataset[gpt_col]]
    convnext_dicts = [json.loads(pred) for pred in hf_dataset[convnext_col]]

    X_gpt = dicts_to_matrix(gpt_dicts, classes)
    X_convnext = dicts_to_matrix(convnext_dicts, classes)
    X = np.hstack([X_gpt, X_convnext])

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    logits = ensemble_model(X_tensor)

    if n_classes > 1:
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    else:
        probs = torch.sigmoid(logits).cpu().numpy()
        probs = np.hstack([1 - probs, probs])

    pred_ids = probs.argmax(axis=1)
    pred_labels = [loaded_id2label[int(i)] for i in pred_ids]

    prob_dicts = [
        {label: float(np.round(p, 6)) for label, p in zip(classes, row)}
        for row in probs
    ]

    return pred_labels, prob_dicts

In [ ]:
"""
Evaluate ensemble on test set and save metrics
"""
from PIL import Image
test_dataset = test_dataset.map(predict_tvm_batch, batched=True, batch_size=1)
test_dataset = test_dataset.map(predict_gpt_batch, batched=True, batch_size=1)

# Access via to_dict() to avoid transform issues
test_dict = test_dataset.to_dict()

# Run ensemble inference using dict data
classes = list(loaded_label2id.keys())
gpt_dicts = [json.loads(pred) for pred in test_dict['prediction_gpt5_allclassprobs']]
convnext_dicts = [json.loads(pred) for pred in test_dict['prediction_convnext_allclassprobs']]

X_gpt = dicts_to_matrix(gpt_dicts, classes)
X_convnext = dicts_to_matrix(convnext_dicts, classes)
X = np.hstack([X_gpt, X_convnext])

X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
with torch.no_grad():
    logits = ensemble_model(X_tensor)

if n_classes > 1:
    probs = torch.softmax(logits, dim=1).cpu().numpy()
else:
    probs = torch.sigmoid(logits).cpu().numpy()
    probs = np.hstack([1 - probs, probs])

pred_ids = probs.argmax(axis=1)
test_preds = [loaded_id2label[int(i)] for i in pred_ids]

y_true = [loaded_id2label[int(i)] for i in test_dict["labels"]]
y_pred = test_preds

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

metrics = {
    "accuracy": float(np.round(accuracy, 6)),
    "precision": float(np.round(precision, 6)),
    "recall": float(np.round(recall, 6)),
    "f1": float(np.round(f1, 6)),
}

os.makedirs(OUT_DIR, exist_ok=True)
metrics_path = os.path.join(OUT_DIR, "test_performance.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)
print(f"Saved test metrics to {metrics_path}")

### Inference
Run the following cells to run your trained ensemble on new unseen data

In [ ]:
"""

Load unseen images for prediction

"""
os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY
client = OpenAI()

import glob
prediction_image_paths = glob.glob('./Example_Dataset/Unseen_Samples/*')
df_predictions = pd.DataFrame(prediction_image_paths, columns=['image_path'])

In [ ]:
"""
Load the saved ensemble model
"""

# Load ensemble checkpoint
ensemble_path = os.path.join(FINAL_ENSEMBLE_MODEL_DIR, "ensemble_model.pt")
checkpoint = torch.load(ensemble_path, map_location=device)

loaded_label2id = checkpoint["label2id"]
loaded_id2label = {v: k for k, v in loaded_label2id.items()}
input_dim = checkpoint["input_dim"]
n_classes = checkpoint["n_classes"]
dropout = checkpoint.get("dropout", 0.25)

ensemble_model = SimpleClassifier(input_dim, n_classes, dropout=dropout).to(device)
ensemble_model.load_state_dict(checkpoint["model_state_dict"])
ensemble_model.eval()

print(f"Loaded ensemble model from {ensemble_path}")

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForImageClassification.from_pretrained(FINAL_MODEL_DIR)
processor = AutoImageProcessor.from_pretrained(FINAL_MODEL_DIR)
model.to(device)
model.eval()

def predict_tvm_batch(examples):
    """Function to predict class probabilities for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        image = Image.open(image_path).convert('RGB')
        
        # Preprocess
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
        
        # logits → probabilities
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        
        # Convert to dictionary with class labels
        class_probs = {model.config.id2label[i]: np.round(float(probs[0, i]), 6) for i in range(probs.shape[1])}
        predictions.append(json.dumps(class_probs))
    
    return {'prediction_convnext_allclassprobs': predictions}

# Get unique classes from the dataset
classes = list(loaded_label2id.keys())
prompt = 'Classify the best fitting class to describe the image by choosing one of the following classes. Return probabilities for all classes in JSON.'

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def get_json_class_schema(classes:list):
    def get_list(classes:list):
        output = {}
        for pred_class in classes:
            output[pred_class] = {
            "type": "number",
            "minimum": 0,
            "maximum": 1
          }
        return output
    class_schema = {
    "format": {
      "type": "json_schema",
      "name": "probabilities",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": get_list(classes),
        "required": classes,
        "additionalProperties": False
      }}}
    return class_schema

def predict_gpt(img_path, prompt, classes):
    base64_image = encode_image(img_path)
    class_schema = get_json_class_schema(classes)

    response = client.responses.create(
        model="gpt-5",
        input=[{
            "role": "system",
            "content": "You are an image classification model. Always return valid JSON with class probabilities that sum to 1."
        },
               {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
        text=class_schema
    )
    
    return response.output_text

def predict_gpt_batch(examples):
    """Function to predict class probabilities using GPT for a batch of images"""
    predictions = []
    
    for image_path in examples['image_path']:
        try:
            response = predict_gpt(image_path, prompt, classes)
            predictions.append(response)
        except Exception as e:
            print(f"Error predicting {image_path}: {e}")
            # Return empty dict in case of error
            predictions.append(json.dumps({c: 0.0 for c in classes}))
    
    return {'prediction_gpt5_allclassprobs': predictions}

# Helper: run ensemble on a HF dataset with prediction columns
@torch.no_grad()
def run_ensemble_inference(hf_dataset, gpt_col="prediction_gpt5_allclassprobs", convnext_col="prediction_convnext_allclassprobs"):
    classes = list(loaded_label2id.keys())

    gpt_dicts = [json.loads(pred) for pred in hf_dataset[gpt_col]]
    convnext_dicts = [json.loads(pred) for pred in hf_dataset[convnext_col]]

    X_gpt = dicts_to_matrix(gpt_dicts, classes)
    X_convnext = dicts_to_matrix(convnext_dicts, classes)
    X = np.hstack([X_gpt, X_convnext])

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    logits = ensemble_model(X_tensor)

    if n_classes > 1:
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    else:
        probs = torch.sigmoid(logits).cpu().numpy()
        probs = np.hstack([1 - probs, probs])

    pred_ids = probs.argmax(axis=1)
    pred_labels = [loaded_id2label[int(i)] for i in pred_ids]

    prob_dicts = [
        {label: float(np.round(p, 6)) for label, p in zip(classes, row)}
        for row in probs
    ]

    return pred_labels, prob_dicts

In [ ]:
"""
Run ensemble inference on df_predictions
"""

# Build a HF dataset from df_predictions and add model probabilities
pred_ds = Dataset.from_pandas(df_predictions)
pred_ds = pred_ds.map(predict_tvm_batch, batched=True, batch_size=1)
pred_ds = pred_ds.map(predict_gpt_batch, batched=True, batch_size=1)

# Access via to_dict() to avoid transform issues
pred_dict = pred_ds.to_dict()

# Run ensemble inference using dict data
classes = list(loaded_label2id.keys())
gpt_dicts = [json.loads(pred) for pred in pred_dict['prediction_gpt5_allclassprobs']]
convnext_dicts = [json.loads(pred) for pred in pred_dict['prediction_convnext_allclassprobs']]

X_gpt = dicts_to_matrix(gpt_dicts, classes)
X_convnext = dicts_to_matrix(convnext_dicts, classes)
X = np.hstack([X_gpt, X_convnext])

X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
with torch.no_grad():
    logits = ensemble_model(X_tensor)

if n_classes > 1:
    probs = torch.softmax(logits, dim=1).cpu().numpy()
else:
    probs = torch.sigmoid(logits).cpu().numpy()
    probs = np.hstack([1 - probs, probs])

pred_ids = probs.argmax(axis=1)
ensemble_preds = [loaded_id2label[int(i)] for i in pred_ids]
ensemble_probs = [
    {label: float(np.round(p, 6)) for label, p in zip(classes, row)}
    for row in probs
]

df_predictions["ensemble_pred"] = ensemble_preds
df_predictions["ensemble_probs"] = [json.dumps(p) for p in ensemble_probs]

# Save results
ensemble_out = os.path.join(OUT_DIR, "prediction_unseen_data_ensemble.csv")
df_predictions.to_csv(ensemble_out, index=False)